In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()
from pyspark.sql.functions import *

In [0]:
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()
data = [
(1, "C001", "Laptop", "50000", "2024-01-01"),
(2, "C002", "Mobile", None, "2024-01-02"),
(4, "C004", "Laptop", "55000", "2024-01-04"),
(5, "C005", "Headphones", None, "2024-01-05"),
(6, "C006", "Camera", "30000", "2024-01-06"),
(7, "C007", "Mobile", "18000", "2024-01-07")]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]


In [0]:
dadf = spark.createDataFrame(data, columns)

### Task 1: Handle NULLs


In [0]:
display(dadf)
dadf.schema

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07


StructType([StructField('order_id', LongType(), True), StructField('customer_id', StringType(), True), StructField('product', StringType(), True), StructField('amount', StringType(), True), StructField('updated_date', StringType(), True)])

### Task 2: Cast Columns
### Convert amount → integer

In [0]:
dadf = dadf.withColumn("amount", dadf.amount.cast("int"))


In [0]:
dadf.filter(dadf.amount.isNull()).display()

order_id,customer_id,product,amount,updated_date
2,C002,Mobile,null,2024-01-02
5,C005,Headphones,null,2024-01-05


### Replace NULL amount with 1000


In [0]:
dadf = dadf.fillna({"amount": 1000})
display(dadf)

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07


### Convert updated_date → date

In [0]:
dadf=dadf.withColumn("date", to_date(col("updated_date")))
display(dadf)

order_id,customer_id,product,amount,updated_date,date
1,C001,Laptop,50000,2024-01-01,2024-01-01
2,C002,Mobile,1000,2024-01-02,2024-01-02
4,C004,Laptop,55000,2024-01-04,2024-01-04
5,C005,Headphones,1000,2024-01-05,2024-01-05
6,C006,Camera,30000,2024-01-06,2024-01-06
7,C007,Mobile,18000,2024-01-07,2024-01-07


### Task 3: Add Derived Columns
### bonus = amount * 0.1

In [0]:
dadf=dadf.withColumn("bonus", col("amount") * 0.1)
display(dadf)

order_id,customer_id,product,amount,updated_date,date,bonus
1,C001,Laptop,50000,2024-01-01,2024-01-01,5000.0
2,C002,Mobile,1000,2024-01-02,2024-01-02,100.0
4,C004,Laptop,55000,2024-01-04,2024-01-04,5500.0
5,C005,Headphones,1000,2024-01-05,2024-01-05,100.0
6,C006,Camera,30000,2024-01-06,2024-01-06,3000.0
7,C007,Mobile,18000,2024-01-07,2024-01-07,1800.0


### category:
### 20000 → High
### else → Low

In [0]:
dadf=dadf.withColumn("category",
    when(col("amount") > 20000, "High")
    .otherwise("Low")
)
display(dadf)

order_id,customer_id,product,amount,updated_date,date,bonus,category
1,C001,Laptop,50000,2024-01-01,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,2024-01-02,100.0,Low
4,C004,Laptop,55000,2024-01-04,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,2024-01-05,100.0,Low
6,C006,Camera,30000,2024-01-06,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,2024-01-07,1800.0,Low


### ### Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High

In [0]:
def grade(amount):
    if amount < 10000:
        return "Low"
    elif amount in (10000,30000):
        return "Medium"
    else:
        return "High"

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

grade_udf = udf(grade, StringType())
dadf = dadf.withColumn("amount_bucket", grade_udf(dadf.amount))
display(dadf)

order_id,customer_id,product,amount,updated_date,date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,2024-01-02,100.0,Low,Low
4,C004,Laptop,55000,2024-01-04,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,2024-01-07,1800.0,Low,High


Task 5: Full Load
### Load all data to target

/Volumes/workspace/default/ene

In [0]:
dadf.write.mode("overwrite").parquet("/Volumes/workspace/default/ene")

### Task 6: Incremental Load
### Load only new/updated records

In [0]:
# incremental loading
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

new_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06"),
    (9, "C009", "Speaker", "15000", "2024-01-08")
]

new_df = spark.createDataFrame(new_data, columns)

# Apply same transformations as dadf
new_df = new_df.withColumn("amount", new_df.amount.cast("int"))
new_df = new_df.fillna({"amount": 1000})
new_df = new_df.withColumn("date", to_date(col("updated_date")))
new_df = new_df.withColumn("bonus", col("amount") * 0.1)
new_df = new_df.withColumn("category", when(col("amount") > 20000, "High").otherwise("Low"))
new_df = new_df.withColumn("amount_bucket", grade_udf(new_df.amount))

dadf = dadf.union(new_df)

last_loaded_date = "2024-01-05"

incremental_df = dadf.filter(col("updated_date") > lit(last_loaded_date))

# Handle duplicates (keep latest record per order_id)
window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

incremental_df = incremental_df.withColumn(
    "rn", row_number().over(window_spec)
).filter(col("rn") == 1).drop("rn")

# Write incremental data
incremental_df.write.mode("append").parquet("/Volumes/workspace/default/ene/")

### Task 7: Parameterization

In [0]:
def run_pipeline(input_path, last_loaded_date):
    df = spark.read.parquet(input_path)

    df = df.withColumn("amount", col("amount").cast(IntegerType())) \
           .withColumn("updated_date", to_date(col("updated_date")))

    df = df.fillna({"amount": "1000"})

    df = df.withColumn("bonus", col("amount") * 0.1)

    df = df.withColumn(
        "category",
        when(col("amount") >= 20000, "High").otherwise("Low")
    )

    df = df.withColumn("amount_bucket", bucket_udf(col("amount")))

    # Incremental logic
    incremental_df = df.filter(col("updated_date") > lit(last_loaded_date))

    window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

    incremental_df = incremental_df.withColumn(
        "rn", row_number().over(window_spec)
    ).filter(col("rn") == 1).drop("rn")

    incremental_df.write.mode("append").parquet("/Volumes/workspace/default/ete/")